# Module 4: Hourly Updates

## Purpose
This module runs hourly to monitor and adjust prices based on:
1. **UTH (Up-Till-Hour) Performance**: Cumulative qty/retailers from start of day until current hour
2. **Last Hour Performance**: Qty/retailers for the most recent hour only

## Schedule
- Runs hourly from 12 PM to 12 AM (midnight), **except** Module 3 hours (12 PM, 3 PM, 6 PM, 9 PM)
- Also runs once at 3 AM
- Active hours: 1 PM, 2 PM, 4 PM, 5 PM, 7 PM, 8 PM, 10 PM, 11 PM, 12 AM, 3 AM

## Data Flow
```
data_extraction.ipynb → Snowflake (Pricing_data_extraction)
                              ↓
                        Module 4 (this module)
                           ├── Load p80_qty, p70_retailers, std columns
                           ├── Fetch fresh: cart rules, stocks, WAC
                           ├── Calculate new_wac (from today's PRS)
                           ├── Query UTH performance (today)
                           ├── Query Last Hour performance (today)
                           ├── Query historical hour contributions
                           ├── Calculate targets & statuses (±1 std)
                           └── Generate actions
```

## New WAC Calculation
To avoid database lag, we calculate a fresh WAC from today's purchase receipts:
- `new_wac1` = ((stocks + reflected_qty) × wac1 + unreflected_qty × item_price) / total_qty
- `new_wac_p` = wac_p × (1 + wac1_change)
- `new_wac` = new_wac_p (with fallback to current_wac)

## Status Logic (±1 Standard Deviation)
SKU status is determined by comparing actual performance to target ± 1 std:
- **Growing**: actual > target + 1×std
- **On Track**: target - 1×std ≤ actual ≤ target + 1×std
- **Dropping**: actual < target - 1×std (minimum threshold = 1)

Standard deviation columns used:
- Qty: `std_daily_240d`
- Retailers: `std_daily_retailers_240d`

## Status Outputs
- `uth_qty_status`: growing / dropping / on_track
- `uth_rets_status`: growing / dropping / on_track
- `last_hour_qty_status`: growing / dropping / on_track
- `last_hour_rets_status`: growing / dropping / on_track


In [ ]:
# =============================================================================
# IMPORTS AND SETUP
# =============================================================================
import pandas as pd
import numpy as np
from datetime import datetime,timedelta
import pytz
import sys
sys.path.append('..')
import setup_environment_2
# Import queries module for Snowflake access
%run queries_module.ipynb

# Cairo timezone
CAIRO_TZ = pytz.timezone('Africa/Cairo')
CAIRO_NOW = datetime.now(CAIRO_TZ)
CURRENT_HOUR = CAIRO_NOW.hour

print(f"Module 4: Hourly Updates")
print(f"Current Cairo Time: {CAIRO_NOW.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Current Hour: {CURRENT_HOUR}")


In [ ]:
# =============================================================================
# CONFIGURATION
# =============================================================================

# Input/Output configuration
# Data is now loaded from Snowflake instead of Excel
INPUT_TABLE = 'MATERIALIZED_VIEWS.Pricing_data_extraction'
OUTPUT_FILE = f'module_4_output_{CAIRO_NOW.strftime("%Y%m%d_%H%M")}.xlsx'

# Status thresholds (±1 std from target)
# - Growing: actual > target + 1*std
# - On Track: target - 1*std <= actual <= target + 1*std
# - Dropping: actual < target - 1*std (minimum threshold = 1)
STD_THRESHOLD = 1  # Number of standard deviations
MIN_DROPPING_THRESHOLD = 1  # Minimum threshold for dropping status
LOW_STOCK_DOH_THRESHOLD = 1  # SKUs with DOH <= this are protected from price reduction

# Qty growing price step limit (max 2 times per day per SKU)
MAX_QTY_GROWING_PRICE_STEPS_PER_DAY = 7

# Cross-module synchronization: 3-hour cooldown window
# Module 4 queries pricing_periodic_push for recent Module 3 increases
# instead of relying on a static hour list (replaced MODULE_3_HOURS)
M3_COOLDOWN_HOURS = 2

print(f"Input: {INPUT_TABLE} (today's data)")
print(f"Output: {OUTPUT_FILE}")
print(f"Status Thresholds: ±{STD_THRESHOLD} std (Dropping minimum = {MIN_DROPPING_THRESHOLD})")


In [ ]:
# =============================================================================
# LOAD PREVIOUS ACTIONS (Track qty_growing_price_step per day)
# =============================================================================
print("Loading previous hourly actions from today...")

def load_previous_hourly_actions():
    """Load previous Module 4 outputs from today to track qty_growing_price_step actions."""
    try:
        query = """
        SELECT product_id, warehouse_id, price_action
        FROM MATERIALIZED_VIEWS.pricing_hourly_push
        WHERE DATE(created_at) = CURRENT_DATE
          AND price_action = 'qty_growing_price_step'
        """
        df = query_snowflake(query)
        return df
    except Exception as e:
        print(f"  Note: Could not load previous actions: {e}")
        return pd.DataFrame()

df_previous_hourly = load_previous_hourly_actions()

# Count qty_growing_price_step actions per SKU today
if len(df_previous_hourly) > 0:
    qty_price_step_counts = (
        df_previous_hourly
        .groupby(['product_id', 'warehouse_id'])
        .size()
        .reset_index(name='qty_price_step_count')
    )
    print(f"Loaded {len(df_previous_hourly)} previous qty_growing_price_step actions")
    print(f"  Unique SKUs with price steps: {len(qty_price_step_counts)}")
else:
    qty_price_step_counts = pd.DataFrame(columns=['product_id', 'warehouse_id', 'qty_price_step_count'])
    print("No previous qty_growing_price_step actions found for today (first run or none triggered)")


In [ ]:
# =============================================================================
# LOAD MODULE 3 RECENT INCREASES (Cross-module synchronization)
# =============================================================================
# Prevent double price increases: if Module 3 already increased a SKU's price
# in the last 3 hours, Module 4 should skip performance-based increases for it.
print("Loading recent Module 3 price increases (last 3 hours)...")

def load_module3_recent_increases():
    """Load SKUs that Module 3 increased in the last M3_COOLDOWN_HOURS hours to avoid double increases."""
    try:
        query = f"""
        SELECT DISTINCT product_id, warehouse_id, 1 as m3_increased_recently
        FROM MATERIALIZED_VIEWS.pricing_periodic_push
        WHERE created_at >= DATEADD(hour, -{M3_COOLDOWN_HOURS}, CURRENT_TIMESTAMP)
          AND (price_action LIKE '%increase%'
               OR (new_price IS NOT NULL AND current_price IS NOT NULL AND new_price > current_price))
        """
        df = query_snowflake(query)
        return df
    except Exception as e:
        print(f"  Note: Could not load Module 3 increases: {e}")
        return pd.DataFrame(columns=['product_id', 'warehouse_id', 'm3_increased_recently'])

df_m3_recent_increases = load_module3_recent_increases()
print(f"  SKUs increased by Module 3 in last 3 hours: {len(df_m3_recent_increases)}")

In [ ]:
# =============================================================================
# LOAD MODULE 4 SELF-COOLDOWN (1-hour minimum between price changes per SKU)
# =============================================================================
# If Module 4 already changed a SKU's price in the last hour, skip it this run.
print("Loading Module 4 recent price changes (last 1 hour)...")

def load_m4_recent_price_changes():
    """Load SKUs where Module 4 changed the price in the last hour."""
    try:
        query = """
        SELECT DISTINCT product_id, warehouse_id, 1 as m4_changed_last_hour
        FROM MATERIALIZED_VIEWS.pricing_hourly_push
        WHERE created_at >= DATEADD(hour, -1, CURRENT_TIMESTAMP)
          AND price_action != 'none'
          AND new_price IS NOT NULL
        """
        df = query_snowflake(query)
        return df
    except Exception as e:
        print(f"  Note: Could not load recent M4 changes: {e}")
        return pd.DataFrame(columns=['product_id', 'warehouse_id', 'm4_changed_last_hour'])

df_m4_recent_changes = load_m4_recent_price_changes()
print(f"  SKUs changed by Module 4 in last 1 hour: {len(df_m4_recent_changes)}")

In [ ]:
# =============================================================================
# LOAD DATA FROM SNOWFLAKE (Instead of Excel file)
# =============================================================================
print("Loading data from Snowflake...")

# Query to get today's data from Pricing_data_extraction
LOAD_QUERY = f"""
SELECT DISTINCT * 
FROM {INPUT_TABLE}
WHERE created_at = (
    SELECT MAX(created_at) 
    FROM {INPUT_TABLE} 
    WHERE created_at >= '{(datetime.now(CAIRO_TZ) - timedelta(days=1)).date()}'
)
"""

df = query_snowflake(LOAD_QUERY)
print(f"Loaded {len(df)} records from Snowflake")

# Early exit if no data (data extraction hasn't run yet for today)
if len(df) == 0:
    print(f"\n⚠️ No data found for today ({datetime.now(CAIRO_TZ).date()})")
    print("Data extraction may not have run yet. Exiting Module 4 gracefully...")
    try:
        sys.path.insert(0, '..')
        from common_functions import send_text_slack
        send_text_slack('new-pricing-logic', f"⚠️ Module 4 skipped at {CAIRO_NOW.strftime('%H:%M')} Cairo - no data for today (data extraction not yet run)")
    except Exception as e:
        print(f"Could not send Slack notification: {e}")
    raise SystemExit("No data available - exiting gracefully")

# Ensure required columns exist with proper types
df['p80_daily_240d'] = pd.to_numeric(df.get('p80_daily_240d', 0), errors='coerce').fillna(0)
df['p70_daily_retailers_240d'] = pd.to_numeric(df.get('p70_daily_retailers_240d', 1), errors='coerce').fillna(1)
df['std_daily_240d'] = pd.to_numeric(df.get('std_daily_240d', 0), errors='coerce').fillna(0)
df['std_daily_retailers_240d'] = pd.to_numeric(df.get('std_daily_retailers_240d', 0), errors='coerce').fillna(0)
df['warehouse_id'] = df['warehouse_id'].astype(int)
df['product_id'] = df['product_id'].astype(int)
df['cohort_id'] = df['cohort_id'].astype(int) if 'cohort_id' in df.columns else None
df['commercial_min_price'] = pd.to_numeric(df.get('commercial_min_price', 0), errors='coerce')
df['commercial_min_price'] = np.round(df['commercial_min_price']*4)/4
# Get category for hourly distribution merge
if 'cat' not in df.columns and 'category' in df.columns:
    df['cat'] = df['category']

print(f"\nP80 Qty Stats: min={df['p80_daily_240d'].min():.1f}, max={df['p80_daily_240d'].max():.1f}, mean={df['p80_daily_240d'].mean():.1f}")
print(f"P70 Retailers Stats: min={df['p70_daily_retailers_240d'].min():.1f}, max={df['p70_daily_retailers_240d'].max():.1f}, mean={df['p70_daily_retailers_240d'].mean():.1f}")
print(f"Std Qty Stats: min={df['std_daily_240d'].min():.1f}, max={df['std_daily_240d'].max():.1f}, mean={df['std_daily_240d'].mean():.1f}")
print(f"Std Retailers Stats: min={df['std_daily_retailers_240d'].min():.1f}, max={df['std_daily_retailers_240d'].max():.1f}, mean={df['std_daily_retailers_240d'].mean():.1f}")

# =============================================================================
# GET FRESH DATA FROM QUERIES MODULE (Snowflake)
# =============================================================================

# 1. Current Cart Rules
df_cart_rules = get_current_cart_rules()

# Merge with main df (by cohort_id + product_id)
if 'cohort_id' in df.columns and len(df_cart_rules) > 0:
    df = df.drop(columns=['current_cart_rule'], errors='ignore')
    df = df.merge(df_cart_rules, on=['cohort_id', 'product_id'], how='left')
    df['current_cart_rule'] = df['current_cart_rule'].fillna(999)
    print(f"✅ Merged cart rules: {len(df)} records")
else:
    df['current_cart_rule'] = df.get('current_cart_rule', 999)

# 2. Current Stocks
df_stocks = get_current_stocks()

# Merge stocks (by warehouse_id + product_id)
if len(df_stocks) > 0:
    df = df.drop(columns=['stocks'], errors='ignore')
    df = df.merge(df_stocks, on=['warehouse_id', 'product_id'], how='left')
    df['stocks'] = df['stocks'].fillna(0)
    print(f"✅ Merged stocks: {len(df)} records")
else:
    df['stocks'] = df.get('stocks', 0)

# 3. Current WAC (Weighted Average Cost)
df_wac = get_current_wac()

# Merge WAC (by warehouse_id + product_id)
if len(df_wac) > 0:
    df = df.drop(columns=['wac_p'], errors='ignore')
    df = df.merge(df_wac, on=['product_id'], how='left')
    df['wac_p'] = df['wac_p'].fillna(0)
    print(f"✅ Merged WAC: {len(df)} records")
else:
    df['wac_p'] = df.get('wac_p', 0)

# 4. Current Prices
df_prices = get_current_prices()

# Merge prices (by cohort_id + product_id)
if len(df_prices) > 0:
    df = df.drop(columns=['current_price'], errors='ignore')
    df = df.merge(df_prices[['cohort_id', 'product_id', 'current_price']], on=['cohort_id', 'product_id'], how='left')
    df['current_price'] = df['current_price'].fillna(0)
    print(f"✅ Merged current prices: {len(df)} records")
else:
    df['current_price'] = df.get('current_price', 0)

print(f"\nCurrent Stock Stats: min={df['stocks'].min():.0f}, max={df['stocks'].max():.0f}, mean={df['stocks'].mean():.1f}")
print(f"Current WAC Stats: min={df['wac_p'].min():.2f}, max={df['wac_p'].max():.2f}, mean={df['wac_p'].mean():.2f}")
print(f"Current Price Stats: min={df['current_price'].min():.2f}, max={df['current_price'].max():.2f}, mean={df['current_price'].mean():.2f}")

# Yesterday's closing stock (proxy for opening stock)
df_closing_stock = get_yesterday_closing_stock()
df = df.drop(columns=['closing_stock_yesterday'], errors='ignore')
df = df.merge(df_closing_stock, on=['warehouse_id', 'product_id'], how='left')
df['closing_stock_yesterday'] = df['closing_stock_yesterday'].fillna(0)
print(f"Yesterday closing stock merged: {(df['closing_stock_yesterday'] > 0).sum()} SKUs had stock at close")

# Quarterly contribution factor for seasonal P80 adjustment
df_qtr_cntrb = get_quarterly_contribution()
df = df.merge(df_qtr_cntrb[['cat', 'qtr_cntrb']], on='cat', how='left')
df['qtr_cntrb'] = df['qtr_cntrb'].fillna(1.0)
print(f"Quarterly contribution merged: min={df['qtr_cntrb'].min():.2f}, max={df['qtr_cntrb'].max():.2f}, mean={df['qtr_cntrb'].mean():.2f}")

# Target turnover qty for high-DOH SKUs
df_target_turnover = get_target_turnover_qty()
df = df.merge(df_target_turnover[['warehouse_id', 'product_id', 'target_qty']], on=['warehouse_id', 'product_id'], how='left')
print(f"Target turnover merged: {df['target_qty'].notna().sum()} high-DOH SKUs have target_qty")

# =============================================================================
# =============================================================================
# LOAD PERCENTILE DATA FOR CART RULES
# =============================================================================
df_percentiles = get_percentile_data()


In [ ]:
# =============================================================================
# CALCULATE NEW WAC (Accounting for Today's Unreflected Purchases)
# =============================================================================
# This calculates a fresh WAC that accounts for today's purchase receipts
# that haven't been reflected in the database yet (to avoid lag)

print("Calculating new WAC from today's purchase receipts...")

# 1. Query today's PRS data (purchases by product_id)
prs_query = '''
WITH prs AS (
    SELECT DISTINCT 
        product_purchased_receipts.purchased_receipt_id,
        products.id AS product_id,
        product_purchased_receipts.basic_unit_count,
        product_purchased_receipts.purchased_item_count * product_purchased_receipts.basic_unit_count AS purchase_min_count,
        product_purchased_receipts.final_price / product_purchased_receipts.purchased_item_count AS final_item_price
    FROM product_purchased_receipts
    LEFT JOIN products ON products.id = product_purchased_receipts.product_id
    LEFT JOIN purchased_receipts ON purchased_receipts.id = product_purchased_receipts.purchased_receipt_id
    WHERE product_purchased_receipts.purchased_item_count <> 0
        AND purchased_receipts.purchased_receipt_status_id IN (4,5,7)
        AND purchased_receipts.date::date = current_date
        AND product_purchased_receipts.final_price > 0 
)
SELECT 
    product_id,
    SUM(purchase_min_count) AS all_day_qty,
    AVG(final_item_price / basic_unit_count) AS item_price
FROM prs
GROUP BY 1
'''
df_prs = setup_environment_2.dwh_pg_query(prs_query, columns=['product_id', 'all_day_qty', 'item_price'])
try:
    df_prs['product_id'] = pd.to_numeric(df_prs['product_id'])
    df_prs['all_day_qty'] = pd.to_numeric(df_prs['all_day_qty'])
    df_prs['item_price'] = pd.to_numeric(df_prs['item_price'])
except:
    df_prs=pd.DataFrame(columns=['product_id', 'all_day_qty', 'item_price'])
print(f"  Loaded {len(df_prs)} PRS records (today's purchases)")

# 2. Query what's already reflected in WAC tracker
reflected_query = '''
SELECT
    t_product_id AS product_id,
    s_beg_stock AS av_stocks,
    p_purchased_item_count AS pr_qty
FROM finance.wac_tracker wt
WHERE wt.t_date::date = CURRENT_DATE
    AND p_purchased_item_count > 0 
'''
try:
    df_reflected = setup_environment_2.dwh_pg_query(reflected_query, columns=['product_id', 'av_stocks', 'pr_qty'])
    df_reflected['product_id'] = pd.to_numeric(df_reflected['product_id'])
    df_reflected['av_stocks'] = pd.to_numeric(df_reflected['av_stocks'])
    df_reflected['pr_qty'] = pd.to_numeric(df_reflected['pr_qty'])
    print(f"  Loaded {len(df_reflected)} WAC tracker records (already reflected)")
except:
    df_reflected = pd.DataFrame(columns=['product_id', 'av_stocks', 'pr_qty'])
    print("  No WAC tracker records found for today")

# 3. Query base WAC (wac1, wac_p) from all_cogs
wac_base_query = '''
SELECT 
    f.product_id,
    f.wac1,
    f.wac_p
FROM finance.all_cogs f
JOIN products ON products.id = f.product_id
JOIN categories ON products.category_id = categories.id
WHERE current_timestamp BETWEEN f.from_date AND f.to_date 
    AND NOT categories.name_ar IN (
        SELECT categories.name_ar AS cat
        FROM categories
        JOIN sections s ON s.id = categories.section_id
        WHERE categories.name_ar LIKE '%سايب%'
            OR categories.name_ar LIKE '%بالتة%'
            OR categories.section_id IN (225, 318, 285, 121, 87, 351, 417)
    )
'''
df_wac_base = query_snowflake(wac_base_query)
df_wac_base['product_id'] = pd.to_numeric(df_wac_base['product_id'])
df_wac_base['wac1'] = pd.to_numeric(df_wac_base['wac1'])
df_wac_base['wac_p'] = pd.to_numeric(df_wac_base['wac_p'])
print(f"  Loaded {len(df_wac_base)} base WAC records from all_cogs")

# 4. Merge and calculate new WAC
df_wac_calc = df_wac_base.merge(df_prs, on='product_id', how='left')
df_wac_calc = df_wac_calc.merge(df_reflected, on='product_id', how='left')
df_wac_calc = df_wac_calc.fillna(0)

# Use current_stock from main df if av_stocks is 0
df_stock_lookup = df[['product_id', 'stocks']].drop_duplicates()
df_stock_lookup = df_stock_lookup.groupby(['product_id'])['stocks'].sum().reset_index()
df_wac_calc = df_wac_calc.merge(df_stock_lookup, on='product_id', how='left')
df_wac_calc['av_stocks'] = df_wac_calc.apply(
    lambda row: row['stocks'] if row['av_stocks'] == 0 else row['av_stocks'], axis=1)

# Calculate not reflected qty (purchases not yet in WAC tracker)
df_wac_calc['not_reflected_qty'] = df_wac_calc['all_day_qty'] - df_wac_calc['pr_qty']
df_wac_calc['not_reflected_qty'] = df_wac_calc['not_reflected_qty'].clip(lower=0)  # Can't be negative

# Calculate new WAC
# new_wac1 = ((stocks + pr_qty) * wac1 + not_reflected_qty * item_price) / (stocks + pr_qty + not_reflected_qty)
denominator = df_wac_calc['av_stocks'] + df_wac_calc['pr_qty'] + df_wac_calc['not_reflected_qty']
numerator = ((df_wac_calc['av_stocks'] + df_wac_calc['pr_qty']) * df_wac_calc['wac1'] + 
             df_wac_calc['not_reflected_qty'] * df_wac_calc['item_price'])

df_wac_calc['new_wac1'] = np.where(denominator > 0, numerator / denominator, df_wac_calc['wac1'])

# Calculate wac1 change and new_wac_p
df_wac_calc['wac1_change'] = np.where(
    df_wac_calc['wac1'] > 0,
    (df_wac_calc['new_wac1'] - df_wac_calc['wac1']) / df_wac_calc['wac1'],
    0
)
df_wac_calc['new_wac_p'] = df_wac_calc['wac_p'] * (1 + df_wac_calc['wac1_change'])
df_wac_calc['new_wac_p'] = df_wac_calc['new_wac_p'].fillna(df_wac_calc['wac_p'])

# Prepare final new_wac dataframe
df_new_wac = df_wac_calc[['product_id','new_wac_p', 'not_reflected_qty']].copy()
df_new_wac=df_new_wac.drop_duplicates()

# 5. Merge new_wac into main dataframe
df = df.drop(columns=['new_wac_p','not_reflected_qty'], errors='ignore')
df = df.merge(df_new_wac, on='product_id', how='left')
df['new_wac'] = df['new_wac_p'].fillna(df['wac_p'])  # Fallback to current_wac if no new_wac

print(f"\n✅ New WAC calculated: {len(df)} records")
print(f"   Products with unreflected purchases: {(df['not_reflected_qty'] > 0).sum()}")
print(f"   New WAC Stats: min={df['new_wac'].min():.2f}, max={df['new_wac'].max():.2f}, mean={df['new_wac'].mean():.2f}")


In [ ]:
# =============================================================================
# QUERY 1: TODAY'S UTH (Up-Till-Hour) PERFORMANCE
# =============================================================================
# Gets cumulative qty and retailers from start of day until current hour
# Uses get_uth_performance() from queries_module


df_uth_today = get_uth_performance()


In [ ]:
# =============================================================================
# QUERY 2: TODAY'S LAST HOUR PERFORMANCE (from DWH)
# =============================================================================
# Gets qty and retailers for the PREVIOUS hour only (not cumulative)
# Uses get_last_hour_performance() from queries_module (DWH/PostgreSQL)

df_last_hour = get_last_hour_performance()


In [ ]:
# =============================================================================
# QUERY 3: HISTORICAL HOURLY DISTRIBUTION (Last 4 Months) - By Category & Warehouse
# =============================================================================
# Gets:
# - avg_uth_pct_qty/retailers: Average contribution of hours 0 to (current_hour-1) to daily total
# - avg_last_hour_pct_qty/retailers: Average contribution of (current_hour-1) alone to daily total
# Uses get_hourly_distribution() from queries_module

df_hourly_dist = get_hourly_distribution()
print(f"\nAvg UTH % (qty): {df_hourly_dist['avg_uth_pct_qty'].mean()*100:.1f}%")
print(f"Avg Last Hour % (qty): {df_hourly_dist['avg_last_hour_pct_qty'].mean()*100:.1f}%")


In [ ]:
# =============================================================================
# MERGE DATA
# =============================================================================
print("Merging performance data with base data...")

# Merge UTH today data
if len(df_uth_today) > 0:
    df = df.merge(df_uth_today, on=['warehouse_id', 'product_id'], how='left')
else:
    df['uth_qty'] = 0
    df['uth_nmv'] = 0
    df['uth_retailers'] = 0

# Merge last hour data
if len(df_last_hour) > 0:
    df = df.merge(df_last_hour, on=['warehouse_id', 'product_id'], how='left')
else:
    df['last_hour_qty'] = 0
    df['last_hour_nmv'] = 0
    df['last_hour_retailers'] = 0

# Merge hourly distribution (by warehouse_id + cat)
if len(df_hourly_dist) > 0:
    df = df.merge(df_hourly_dist, on=['warehouse_id', 'cat'], how='left')
else:
    df['avg_uth_pct_qty'] = 0.5
    df['avg_uth_pct_retailers'] = 0.5
    df['avg_last_hour_pct_qty'] = 0.05
    df['avg_last_hour_pct_retailers'] = 0.05

# Fill NaN values
df['uth_qty'] = df['uth_qty'].fillna(0)
df['uth_nmv'] = df['uth_nmv'].fillna(0)
df['uth_retailers'] = df['uth_retailers'].fillna(0)
df['last_hour_qty'] = df['last_hour_qty'].fillna(0)
df['last_hour_nmv'] = df['last_hour_nmv'].fillna(0)
df['last_hour_retailers'] = df['last_hour_retailers'].fillna(0)
df['avg_uth_pct_qty'] = df['avg_uth_pct_qty'].fillna(0.5)
df['avg_uth_pct_retailers'] = df['avg_uth_pct_retailers'].fillna(0.5)
df['avg_last_hour_pct_qty'] = df['avg_last_hour_pct_qty'].fillna(0.05)
df['avg_last_hour_pct_retailers'] = df['avg_last_hour_pct_retailers'].fillna(0.05)

print(f"✅ Merged data: {len(df)} records")
print(f"\nUTH Qty Stats: min={df['uth_qty'].min():.0f}, max={df['uth_qty'].max():.0f}, mean={df['uth_qty'].mean():.1f}")
print(f"Last Hour Qty Stats: min={df['last_hour_qty'].min():.0f}, max={df['last_hour_qty'].max():.0f}, mean={df['last_hour_qty'].mean():.1f}")
print(f"Current Cart Rule Stats: min={df['current_cart_rule'].min():.0f}, max={df['current_cart_rule'].max():.0f}, mean={df['current_cart_rule'].mean():.1f}")


In [ ]:
# =============================================================================
# CALCULATE TARGETS AND STATUSES
# =============================================================================
print("Calculating UTH and Last Hour targets and statuses...")

def get_status_std(actual, target, std, is_qty=True):
    """
    Determine status based on ±STD_THRESHOLD std from target.
    - Growing: actual > target + STD_THRESHOLD*std
    - On Track: lower_bound <= actual <= upper_bound
    - Dropping: actual < target - STD_THRESHOLD*std (minimum threshold = 1)
    """
    upper_bound = target + STD_THRESHOLD * std
    lower_bound = max(target - STD_THRESHOLD * std, MIN_DROPPING_THRESHOLD)
    if actual > upper_bound:
        return 'growing'
    elif actual < lower_bound:
        return 'dropping'
    else:
        return 'on_track'

# Apply quarterly seasonal adjustment to P80
df['p80_daily_240d_adj'] = df['p80_daily_240d'] * df['qtr_cntrb']

# Turnover-based target for high-DOH SKUs (from target turnover query, scaled by qtr_cntrb)
df['turnover_daily'] = pd.to_numeric(df['target_qty'], errors='coerce').fillna(0) * df['qtr_cntrb']

# Calculate UTH targets: max of P80-based target and turnover target (both scaled by UTH %)
df['uth_qty_target'] = np.maximum(
    df['p80_daily_240d_adj'] * df['avg_uth_pct_qty'],
    df['turnover_daily'] * df['avg_uth_pct_qty']
).clip(lower=2)
df['uth_rets_target'] = df['p70_daily_retailers_240d'] * df['avg_uth_pct_retailers']

# Calculate UTH std thresholds (scaled by UTH percentage)
df['uth_qty_std'] = df['std_daily_240d'] * df['avg_uth_pct_qty']
df['uth_rets_std'] = df['std_daily_retailers_240d'] * df['avg_uth_pct_retailers']

# Calculate Last Hour targets: max of P80-based target and turnover target (both scaled by last hour %)
df['last_hour_qty_target'] = np.maximum(
    df['p80_daily_240d_adj'] * df['avg_last_hour_pct_qty'],
    df['turnover_daily'] * df['avg_last_hour_pct_qty']
).clip(lower=2)
df['last_hour_rets_target'] = df['p70_daily_retailers_240d'] * df['avg_last_hour_pct_retailers']

# Calculate Last Hour std thresholds (scaled by last hour percentage)
df['last_hour_qty_std'] = df['std_daily_240d'] * df['avg_last_hour_pct_qty']
df['last_hour_rets_std'] = df['std_daily_retailers_240d'] * df['avg_last_hour_pct_retailers']

# Calculate statuses using ±3 std thresholds
df['uth_qty_status'] = df.apply(
    lambda row: get_status_std(row['uth_qty'], row['uth_qty_target'], row['uth_qty_std'], is_qty=True), axis=1)
df['uth_rets_status'] = df.apply(
    lambda row: get_status_std(row['uth_retailers'], row['uth_rets_target'], row['uth_rets_std'], is_qty=False), axis=1)
df['last_hour_qty_status'] = df.apply(
    lambda row: get_status_std(row['last_hour_qty'], row['last_hour_qty_target'], row['last_hour_qty_std'], is_qty=True), axis=1)
df['last_hour_rets_status'] = df.apply(
    lambda row: get_status_std(row['last_hour_retailers'], row['last_hour_rets_target'], row['last_hour_rets_std'], is_qty=False), axis=1)

print(f"✅ Targets and statuses calculated using ±{STD_THRESHOLD} std thresholds")

# Summary
print(f"\n{'='*60}")
print("UTH STATUS DISTRIBUTION")
print(f"{'='*60}")
print(f"\nUTH Qty Status:")
print(df['uth_qty_status'].value_counts().to_string())
print(f"\nUTH Retailers Status:")
print(df['uth_rets_status'].value_counts().to_string())

print(f"\n{'='*60}")
print("LAST HOUR STATUS DISTRIBUTION")
print(f"{'='*60}")
print(f"\nLast Hour Qty Status:")
print(df['last_hour_qty_status'].value_counts().to_string())
print(f"\nLast Hour Retailers Status:")
print(df['last_hour_rets_status'].value_counts().to_string())


In [ ]:
# =============================================================================
# FETCH MARKET DATA AND MARGIN TIERS
# =============================================================================
print("Fetching market data and margin tiers...")

# Import market_data_module for get_market_data()
%run market_data_module.ipynb

# 1. Get margin tiers (already available from queries_module)
df_margin_tiers = get_margin_tiers()
print(f"✅ Loaded {len(df_margin_tiers)} margin tier records")

# 2. Get market data
df_market = get_market_data()
print(f"✅ Loaded {len(df_market)} market data records")

# Verify cohort_id is available in all dataframes
print(f"\nChecking cohort_id availability:")
print(f"  - df has cohort_id: {'cohort_id' in df.columns}")
print(f"  - df_margin_tiers has cohort_id: {'cohort_id' in df_margin_tiers.columns}")
print(f"  - df_market has cohort_id: {'cohort_id' in df_market.columns}")

# 3. Merge margin tiers with df (by warehouse_id + product_id) - ALL COLUMNS
merge_keys = ['warehouse_id', 'product_id']
df = df.drop(columns=[c for c in df_margin_tiers.columns if c in df.columns and c not in merge_keys], errors='ignore')
df = df.merge(df_margin_tiers, on=merge_keys, how='left')
print(f"\n✅ Merged margin tiers: {len(df)} records")
print(f"   Margin tier columns added: {[c for c in df_margin_tiers.columns if c not in merge_keys]}")

# 4. Merge market data with df (by cohort_id + product_id) - ALL COLUMNS
market_merge_keys = ['cohort_id', 'product_id']
df = df.drop(columns=[c for c in df_market.columns if c in df.columns and c not in market_merge_keys], errors='ignore')
df = df.drop(columns=['market_data_source'], errors='ignore')
df = df.merge(df_market, on=market_merge_keys, how='left')
print(f"✅ Merged market data: {len(df)} records")
print(f"   Market columns added: {[c for c in df_market.columns if c not in market_merge_keys]}")

# 4b. Apply brand market percentile fallback for SKUs without market data
print("Applying brand market percentile fallback...")
df_brand_percs = get_brand_market_percentiles()
df = fill_brand_market_fallback(df, df_brand_percs)
print(f"   Market data source distribution: {df['market_data_source'].value_counts(dropna=False).to_dict()}")

# 5. Calculate current margin based on current_price and new_wac
df['current_margin'] = (df['current_price'] - df['new_wac']) / df['current_price']
df['current_margin'] = df['current_margin'].fillna(0)

print(f"\nMargin Stats: min={df['current_margin'].min():.2%}, max={df['current_margin'].max():.2%}, mean={df['current_margin'].mean():.2%}")

# =============================================================================
# FETCH HIGH DOH SKUS FROM MODULE 3 OUTPUT
# =============================================================================
# SKUs with high DOH should NOT have price increases (only price decreases in Module 3)
print("Fetching high DOH SKUs from Module 3...")

HIGH_DOH_QUERY = '''
SELECT DISTINCT product_id, warehouse_id, 1 as is_high_doh
FROM MATERIALIZED_VIEWS.pricing_periodic_push
WHERE created_at::DATE = CURRENT_DATE
  AND (uth_status = 'High DOH' OR price_action LIKE '%doh%')
'''
try:
    df_high_doh = query_snowflake(HIGH_DOH_QUERY)
    print(f"  Loaded {len(df_high_doh)} high DOH SKUs from Module 3")
except:
    df_high_doh = pd.DataFrame(columns=['product_id', 'warehouse_id', 'is_high_doh'])
    print("  No high DOH data found (Module 3 may not have run yet)")

# Merge high DOH flag
df['is_high_doh'] = 0  # Initialize column first
if len(df_high_doh) > 0:
    df = df.drop(columns=['is_high_doh'], errors='ignore')
    df = df.merge(df_high_doh[['product_id', 'warehouse_id', 'is_high_doh']], 
                  on=['product_id', 'warehouse_id'], how='left')
    df['is_high_doh'] = df['is_high_doh'].fillna(0).astype(int)
print(f"  SKUs marked as high DOH: {(df['is_high_doh'] == 1).sum()}")

# Merge Module 3 recent increases flag (cross-module synchronization)
df['m3_increased_recently'] = 0
if len(df_m3_recent_increases) > 0:
    df = df.drop(columns=['m3_increased_recently'], errors='ignore')
    df = df.merge(df_m3_recent_increases[['product_id', 'warehouse_id', 'm3_increased_recently']], 
                  on=['product_id', 'warehouse_id'], how='left')
    df['m3_increased_recently'] = df['m3_increased_recently'].fillna(0).astype(int)
print(f"  SKUs blocked by Module 3 recent increase (3h cooldown): {(df['m3_increased_recently'] == 1).sum()}")

# Merge Module 4 self-cooldown flag (1-hour minimum between price changes)
df['m4_changed_last_hour'] = 0
if len(df_m4_recent_changes) > 0:
    df = df.drop(columns=['m4_changed_last_hour'], errors='ignore')
    df = df.merge(df_m4_recent_changes[['product_id', 'warehouse_id', 'm4_changed_last_hour']], 
                  on=['product_id', 'warehouse_id'], how='left')
    df['m4_changed_last_hour'] = df['m4_changed_last_hour'].fillna(0).astype(int)
print(f"  SKUs blocked by Module 4 self-cooldown (1h): {(df['m4_changed_last_hour'] == 1).sum()}")

# =============================================================================
# FILTER SKUS FOR ACTION
# =============================================================================
# Filter SKUs that need action based on:
# Pre-condition: below_min_stock_flag != 1 (must be sellable)
# Condition A: new_wac > wac_p * 1.005 (WAC increased by more than 0.5%)
# Condition B: Last hour growing AND UTH healthy AND NOT high DOH AND NOT recently increased by M3 AND NOT changed by M4 last hour

print("Filtering SKUs for action...")

# Skip SKUs with below_min_stock_flag (stock < min selling unit qty)
can_sell = df['below_min_stock_flag'].fillna(0) != 1

# Condition A: WAC increased significantly
condition_a = (df['new_wac'] > df['wac_p'] * 1.005)

# Condition B: UTH healthy (growing or on_track) AND last hour spiking AND not high DOH AND not recently increased by Module 3
uth_healthy = df['uth_qty_status'].isin(['growing', 'on_track'])
last_hour_spiking = df['last_hour_qty_status'] == 'growing'
not_high_doh = df['is_high_doh'] != 1
not_recently_increased_m3 = df['m3_increased_recently'] != 1
not_changed_last_hour = df['m4_changed_last_hour'] != 1
condition_b = uth_healthy & last_hour_spiking & can_sell & not_high_doh & not_recently_increased_m3 & not_changed_last_hour

# Condition C: Current price below commercial minimum (must enforce commercial min)
condition_c = (
    (df['commercial_min_price'].notna()) & 
    (df['commercial_min_price'] > 0) & 
    (df['current_price'] < df['commercial_min_price'])
)

# Combine conditions (A OR B OR C)
df['needs_action'] = condition_a | condition_b | condition_c
df['action_reason'] = 'none'
df.loc[condition_a & ~condition_b & ~condition_c, 'action_reason'] = 'wac_increase'
df.loc[~condition_a & condition_b & ~condition_c, 'action_reason'] = 'growing_performance'
df.loc[~condition_a & ~condition_b & condition_c, 'action_reason'] = 'below_commercial_min'
df.loc[condition_a & condition_b & ~condition_c, 'action_reason'] = 'both'
df.loc[condition_a & ~condition_b & condition_c, 'action_reason'] = 'wac_and_commercial'
df.loc[~condition_a & condition_b & condition_c, 'action_reason'] = 'growing_and_commercial'
df.loc[condition_a & condition_b & condition_c, 'action_reason'] = 'all'

# Define important columns for df_action
action_columns = [
    # Identification
    'warehouse_id', 'product_id', 'cohort_id', 'sku', 'region', 'brand', 'cat',
    # Price & Cost
    'current_price', 'wac_p', 'new_wac', 'current_margin', 'commercial_min_price',
    # Inventory & Rules
    'stocks', 'current_cart_rule', 'doh',
    # Performance Targets
    'p80_daily_240d', 'p70_daily_retailers_240d', 'std_daily_240d', 'std_daily_retailers_240d', 'qtr_cntrb',
    # Refill columns for cart rule calculation
    'normal_refill', 'refill_stddev', 'target_margin',
    # UTH Status
    'uth_qty', 'uth_qty_target', 'uth_qty_status', 'uth_retailers', 'uth_rets_status',
    # Last Hour Status
    'last_hour_qty', 'last_hour_qty_status', 'last_hour_retailers', 'last_hour_rets_status',
    # Market Margins
    'below_market', 'market_min', 'market_25', 'market_50', 'market_75', 'market_max', 'above_market',
    # Margin Tiers (excluding: effective_min_margin, max_boundary, margin_step, margin_tier_below)
    'margin_tier_1', 'margin_tier_2', 'margin_tier_3', 'margin_tier_4', 'margin_tier_5',
    'margin_tier_above_1', 'margin_tier_above_2',
    # Action Flags
    'needs_action', 'action_reason'
]

# Filter to only columns that exist in df
action_columns = [c for c in action_columns if c in df.columns]

# Filter to action SKUs with selected columns only
df_action = df[df['needs_action']][action_columns].copy()

# Merge previous qty_growing_price_step counts
df_action = df_action.merge(qty_price_step_counts, on=['product_id', 'warehouse_id'], how='left')
df_action['qty_price_step_count'] = df_action['qty_price_step_count'].fillna(0).astype(int)

print(f"\n{'='*60}")
print("SKUs FILTERED FOR ACTION")
print(f"{'='*60}")
print(f"\nTotal SKUs needing action: {len(df_action)} out of {len(df)} ({len(df_action)/len(df)*100:.1f}%)")
print(f"\nBreakdown by reason:")
print(df_action['action_reason'].value_counts().to_string())

print(f"\n--- Condition A (WAC increase > 0.5%): {condition_a.sum()} SKUs")
print(f"--- Condition B (UTH healthy + last hour spiking + not high DOH + no M3 cooldown + no M4 self-cooldown): {condition_b.sum()} SKUs")
print(f"--- Condition C (Below commercial minimum): {condition_c.sum()} SKUs")
print(f"--- High DOH SKUs blocked from growing action: {(df['is_high_doh'] == 1).sum()}")
print(f"--- Module 3 cooldown SKUs blocked from growing action: {(df['m3_increased_recently'] == 1).sum()}")
print(f"--- Module 4 self-cooldown SKUs blocked (changed last hour): {(df['m4_changed_last_hour'] == 1).sum()}")

# Show sample
print(f"\n{'='*60}")
print("SAMPLE ACTION SKUs")
print(f"{'='*60}")
print(f"\nTotal columns in df_action: {len(df_action.columns)}")
print(f"Columns: {df_action.columns.tolist()}")
display(df_action.head(10))


In [ ]:
# =============================================================================
# ACTION LOGIC
# =============================================================================
print("Calculating actions for filtered SKUs...")

# Step 0: Preparation - Calculate normal_refill_3std
df_action['normal_refill_3std'] = df_action['normal_refill'] + 3 * df_action['refill_stddev']

# Initialize output columns
df_action['new_price'] = np.nan
df_action['new_cart_rule'] = np.nan
df_action['price_action'] = 'none'
df_action['cart_rule_action'] = 'none'

# =============================================================================
# HELPER: Calculate all price tiers from margins using wac_p
# =============================================================================
def calculate_price_from_margin(wac, margin):
    """Calculate price from WAC and margin: price = wac / (1 - margin)"""
    if pd.isna(margin) or margin >= 1:
        return np.nan
    return wac / (1 - margin)

# Calculate market prices (using wac_p)
market_margin_cols = ['market_min', 'market_25', 'market_50', 'market_75', 'market_max']
for col in market_margin_cols:
    price_col = f'{col}_price'
    df_action[price_col] = df_action.apply(
        lambda row: calculate_price_from_margin(row['wac_p'], row[col]), axis=1)

# Calculate margin tier prices (using wac_p)
tier_margin_cols = ['margin_tier_1', 'margin_tier_2', 'margin_tier_3', 'margin_tier_4', 'margin_tier_5',
                    'margin_tier_above_1', 'margin_tier_above_2']
for col in tier_margin_cols:
    price_col = f'{col}_price'
    df_action[price_col] = df_action.apply(
        lambda row: calculate_price_from_margin(row['wac_p'], row[col]), axis=1)

# =============================================================================
# HELPER: Subdivide large tier gaps
# =============================================================================
def subdivide_tiers(price_tiers, wac, target_margin, max_gap_pct=0.30):
    """Recursively insert midpoints between tiers with margin gaps > max_gap_pct * target_margin."""
    if wac <= 0 or target_margin <= 0 or len(price_tiers) < 2:
        return price_tiers
    
    max_margin_gap = target_margin * max_gap_pct
    result = sorted(set(price_tiers))
    
    while True:
        new_result = [result[0]]
        for i in range(1, len(result)):
            m_prev = (result[i-1] - wac) / result[i-1] if result[i-1] > 0 else 0
            m_curr = (result[i] - wac) / result[i] if result[i] > 0 else 0
            if abs(m_curr - m_prev) > max_margin_gap:
                mid_margin = (m_prev + m_curr) / 2
                if 0 < mid_margin < 1:
                    mid_price = round(wac / (1 - mid_margin) * 4) / 4
                    new_result.append(mid_price)
            new_result.append(result[i])
        new_result = sorted(set(new_result))
        if new_result == result:
            break
        result = new_result
    
    return result


# HELPER: Find first price above threshold (with gap interpolation)
# =============================================================================
def find_first_price_above(row, threshold, price_cols):
    """Find the first enriched price tier above threshold. Subdivides gaps within the given price_cols set."""
    prices = []
    for col in price_cols:
        price = row.get(col, np.nan)
        if pd.notna(price) and price > 0:
            prices.append(price)
    
    if not prices:
        return np.nan
    
    wac = row.get('wac_p', 0) or 0
    target_margin = row.get('target_margin', 0) or 0
    enriched = subdivide_tiers(sorted(set(prices)), wac, target_margin)
    
    for tier in enriched:
        if tier > threshold:
            return tier
    return np.nan

# Define price column order (market first, then tiers)
market_price_cols = ['market_min_price', 'market_25_price', 'market_50_price', 'market_75_price', 'market_max_price']
tier_price_cols = ['margin_tier_1_price', 'margin_tier_2_price', 'margin_tier_3_price', 
                   'margin_tier_4_price', 'margin_tier_5_price', 
                   'margin_tier_above_1_price', 'margin_tier_above_2_price']
all_price_cols = market_price_cols + tier_price_cols

def smooth_price_increase(row, next_tier_price):
    """Half-step toward next tier, clamped by min/max margin step bounds."""
    current_price = row['current_price']
    wac = row.get('wac_p', 0)
    target_margin = row.get('target_margin', 0) or 0

    if pd.isna(next_tier_price) or wac <= 0 or current_price <= 0:
        return np.nan

    current_margin = (current_price - wac) / current_price

    half_price = current_price + (next_tier_price - current_price) / 2
    half_margin = (half_price - wac) / half_price
    margin_step = half_margin - current_margin

    min_step = max(target_margin * 0.08, 0.0025)
    max_step = min(target_margin * 0.60, 0.03) if target_margin > 0 else 0.03
    min_step = min(min_step, max_step)

    clamped_step = max(min_step, min(margin_step, max_step))

    new_margin = current_margin + clamped_step
    if new_margin >= 1:
        return np.nan

    new_price = wac / (1 - new_margin)
    return round(new_price * 4) / 4

# =============================================================================
# STEP 1: WAC Increase Actions (action_reason includes 'wac_increase')
# =============================================================================
# Trigger when new_wac causes current margin to drop below margin_tier_1.
# Market-first: find first market price above new_wac, else margin tiers with tier_1 floor.
wac_increase_mask = (
    df_action['action_reason'].isin(['both', 'wac_increase', 'wac_and_commercial', 'all'])
)

def get_wac_increase_price(row):
    """Market-first WAC fix: first market price above new_wac, else margin tier with tier_1 floor."""
    new_wac = row['new_wac']
    current_price = row['current_price']
    margin_tier_1 = row.get('margin_tier_1', np.nan)

    if pd.isna(margin_tier_1) or current_price <= 0 or new_wac <= 0:
        return np.nan

    current_margin_with_new_wac = (current_price - new_wac) / current_price

    if current_margin_with_new_wac < margin_tier_1:
        tier_1_price = new_wac / (1 - margin_tier_1) if margin_tier_1 < 1 else np.nan

        # Priority 1: Market price above new_wac
        market_price = find_first_price_above(row, new_wac, market_price_cols)
        if pd.notna(market_price):
            smoothed = smooth_price_increase(row, market_price)
            if pd.notna(smoothed) and pd.notna(tier_1_price):
                return max(smoothed, round(tier_1_price * 4) / 4)
            if pd.notna(tier_1_price):
                return max(round(market_price * 4) / 4, round(tier_1_price * 4) / 4)
            return round(market_price * 4) / 4

        # Priority 2: Margin tier fallback (floor at margin_tier_1)
        if pd.notna(tier_1_price):
            next_tier = find_first_price_above(row, tier_1_price, tier_price_cols)
            if pd.notna(next_tier):
                smoothed = smooth_price_increase(row, next_tier)
                if pd.notna(smoothed):
                    return max(smoothed, round(tier_1_price * 4) / 4)
            return round(tier_1_price * 4) / 4
    return np.nan

df_action.loc[wac_increase_mask, 'new_price'] = df_action[wac_increase_mask].apply(get_wac_increase_price, axis=1)
df_action.loc[wac_increase_mask, 'price_action'] = 'wac_increase'

print(f"  WAC increase actions: {wac_increase_mask.sum()} SKUs")
print(f"    (SKUs where new_wac pushes margin below margin_tier_1, market-first)")

# =============================================================================
# STEP 2: Growing Performance Actions (action_reason = 'growing_performance')
# =============================================================================
growing_mask = df_action['action_reason'] == 'growing_performance'

# Case A: Retailers growing (with or without qty)
rets_growing_mask = growing_mask & (df_action['last_hour_rets_status'] == 'growing')

def get_rets_growing_price(row):
    """Find price for rets growing: smoothed half-step toward next market/tier price."""
    min_price = row['current_price'] * 1.005
    
    # Try market prices first
    next_price = find_first_price_above(row, min_price, market_price_cols)
    if pd.notna(next_price):
        return smooth_price_increase(row, next_price)
    
    # Try tier prices
    next_price = find_first_price_above(row, min_price, tier_price_cols)
    if pd.notna(next_price):
        return smooth_price_increase(row, next_price)
    
    # Fallback: target margin (already a small step, no smoothing needed)
    target_margin = row.get('target_margin', 0)
    if pd.notna(target_margin) and target_margin < 1:
        fallback_price = row['wac_p'] / (1 - (target_margin))
        if fallback_price < row['current_price']:
            return row['current_price'] * 1.01
        return fallback_price
    
    return np.nan

df_action.loc[rets_growing_mask, 'new_price'] = df_action[rets_growing_mask].apply(get_rets_growing_price, axis=1)
df_action.loc[rets_growing_mask, 'price_action'] = 'rets_growing'

print(f"  Rets growing actions (price increase): {rets_growing_mask.sum()} SKUs")

# Case B: Qty growing only (rets NOT growing)
qty_only_growing_mask = growing_mask & (df_action['last_hour_qty_status'] == 'growing') & (df_action['last_hour_rets_status'] != 'growing')

def get_current_percentile_level(current_cart_rule, percentile_row):
    """Determine which percentile level current cart rule corresponds to."""
    if len(percentile_row) == 0:
        return None
    
    perc_95 = percentile_row.iloc[0]['perc_95']
    perc_75 = percentile_row.iloc[0]['perc_75']
    perc_50 = percentile_row.iloc[0]['perc_50']
    perc_25 = percentile_row.iloc[0]['perc_25']
    
    # Determine current level (with tolerance for rounding)
    if pd.notna(perc_95) and abs(current_cart_rule - perc_95) <= 2:
        return 95
    elif pd.notna(perc_75) and abs(current_cart_rule - perc_75) <= 2:
        return 75
    elif pd.notna(perc_50) and abs(current_cart_rule - perc_50) <= 2:
        return 50
    elif pd.notna(perc_25) and abs(current_cart_rule - perc_25) <= 2:
        return 25
    return None

def get_next_lower_percentile(current_level, percentile_row):
    """Get next lower percentile value."""
    if len(percentile_row) == 0:
        return None
    
    if current_level == 95:
        return percentile_row.iloc[0]['perc_75']
    elif current_level == 75:
        return percentile_row.iloc[0]['perc_50']
    elif current_level == 50:
        return percentile_row.iloc[0]['perc_25']
    elif current_level == 25:
        return percentile_row.iloc[0]['perc_25']  # Stay at minimum
    return None

def get_qty_growing_cart_rule(row):
    """Reduce cart rule by one percentile level when qty is spiking (>1.5x target)."""
    current_cart = row.get('current_cart_rule', 5)

    uth_qty = row.get('uth_qty', 0)
    uth_qty_target = row.get('uth_qty_target', 1)
    qty_ratio = uth_qty / uth_qty_target if uth_qty_target > 0 else 0

    if qty_ratio > 2:
        cohort_id = row.get('cohort_id')
        product_id = row.get('product_id')
        percentile_row = df_percentiles[
            (df_percentiles['cohort_id'] == cohort_id) & 
            (df_percentiles['product_id'] == product_id)
        ]

        if len(percentile_row) > 0:
            current_level = get_current_percentile_level(current_cart, percentile_row)
            if current_level:
                next_perc = get_next_lower_percentile(current_level, percentile_row)
                if pd.notna(next_perc) and next_perc > 0:
                    return max(5, min(500, int(round(next_perc))))

    return current_cart

df_action.loc[qty_only_growing_mask, 'new_cart_rule'] = df_action[qty_only_growing_mask].apply(get_qty_growing_cart_rule, axis=1)

# Round and apply min/max constraints to new_cart_rule
df_action['new_cart_rule'] = df_action['new_cart_rule'].round()
df_action['new_cart_rule'] = df_action['new_cart_rule'].clip(lower=5, upper=500)

# LOW STOCK PROTECTION: Cap cart rule at normal_refill for low stock SKUs with demand
# This prevents cart expansion that could deplete inventory before next receiving
if 'doh' in df_action.columns:
    low_stock_mask = (
        (df_action['doh'].fillna(999) <= LOW_STOCK_DOH_THRESHOLD) & 
        (df_action['uth_qty'].fillna(0) > 0) &
        (df_action['new_cart_rule'].notna())
    )
    if low_stock_mask.sum() > 0:
        # Cap cart rule at normal_refill for low stock SKUs
        df_action.loc[low_stock_mask, 'new_cart_rule'] = df_action.loc[low_stock_mask].apply(
            lambda row: min(row['new_cart_rule'], np.ceil(row.get('normal_refill', row['new_cart_rule']))), axis=1
        )
        print(f"  Low stock protection: {low_stock_mask.sum()} SKUs had cart rule capped at normal_refill")

df_action.loc[qty_only_growing_mask, 'cart_rule_action'] = 'qty_growing'

print(f"  Qty growing actions (cart rule + price): {qty_only_growing_mask.sum()} SKUs")
print(f"    (Cart rule: rounded, min=5, max=500)")

# Qty growing also gets a smooth price increase (market-first, same as rets)
can_do_price_step = df_action['qty_price_step_count'] < MAX_QTY_GROWING_PRICE_STEPS_PER_DAY
qty_growing_price_mask = qty_only_growing_mask & can_do_price_step

def get_qty_growing_price(row):
    """Market-first smooth price increase for qty growing."""
    min_price = row['current_price'] * 1.005

    # Priority 1: Market prices
    next_price = find_first_price_above(row, min_price, market_price_cols)
    if pd.notna(next_price):
        return smooth_price_increase(row, next_price)

    # Priority 2: Tier prices
    next_price = find_first_price_above(row, min_price, tier_price_cols)
    if pd.notna(next_price):
        return smooth_price_increase(row, next_price)

    return np.nan

df_action.loc[qty_growing_price_mask, 'new_price'] = df_action[qty_growing_price_mask].apply(get_qty_growing_price, axis=1)
df_action.loc[qty_growing_price_mask, 'price_action'] = 'qty_growing_price_step'

qty_growing_blocked_limit = qty_only_growing_mask & ~can_do_price_step
print(f"  Qty growing with price step: {qty_growing_price_mask.sum()} SKUs")
print(f"  Qty growing blocked (already {MAX_QTY_GROWING_PRICE_STEPS_PER_DAY}x today): {qty_growing_blocked_limit.sum()} SKUs")

# =============================================================================
# FINAL STEP: ENFORCE COMMERCIAL MINIMUM PRICE
# =============================================================================
# For all SKUs in df_action, ensure new_price >= commercial_min_price
print(f"\n{'='*60}")
print("ENFORCING COMMERCIAL MINIMUM PRICE")
print(f"{'='*60}")

has_commercial_min = (df_action['commercial_min_price'].notna()) & (df_action['commercial_min_price'] > 0)

# For SKUs with new_price already calculated (growing/wac): MAX(new_price, commercial_min)
has_new_price = df_action['new_price'].notna()
below_min_with_new = has_commercial_min & has_new_price & (df_action['new_price'] < df_action['commercial_min_price'])
df_action.loc[below_min_with_new, 'new_price'] = df_action.loc[below_min_with_new, 'commercial_min_price']
df_action.loc[below_min_with_new, 'price_action'] = df_action.loc[below_min_with_new, 'price_action'].astype(str) + '_commercial_enforced'

# For SKUs without new_price (only commercial min violation): new_price = commercial_min
no_new_price = df_action['new_price'].isna()
below_min_only = has_commercial_min & no_new_price & (df_action['current_price'] < df_action['commercial_min_price'])
df_action.loc[below_min_only, 'new_price'] = df_action.loc[below_min_only, 'commercial_min_price']
df_action.loc[below_min_only, 'price_action'] = 'commercial_min_enforced'

print(f"  SKUs with commercial min: {has_commercial_min.sum()}")
print(f"  Growing SKUs with commercial min applied: {below_min_with_new.sum()}")
print(f"  SKUs only needing commercial min: {below_min_only.sum()}")
print(f"  Total commercial min enforcements: {below_min_with_new.sum() + below_min_only.sum()}")

# =============================================================================
# SUMMARY
# =============================================================================
print(f"\n{'='*60}")
print("ACTION SUMMARY")
print(f"{'='*60}")
print(f"\nPrice Actions:")
print(df_action['price_action'].value_counts().to_string())
print(f"\nCart Rule Actions:")
print(df_action['cart_rule_action'].value_counts().to_string())

# Show SKUs with new prices
price_changes = df_action[df_action['new_price'].notna()]
print(f"\nSKUs with new price: {len(price_changes)}")
if len(price_changes) > 0:
    print(f"  Avg price change: {((price_changes['new_price'] - price_changes['current_price']) / price_changes['current_price']).mean()*100:.1f}%")

# Show SKUs with new cart rules
cart_changes = df_action[df_action['new_cart_rule'].notna()]
print(f"\nSKUs with new cart rule: {len(cart_changes)}")
if len(cart_changes) > 0:
    print(f"  Avg cart rule change: {((cart_changes['new_cart_rule'] - cart_changes['current_cart_rule']) / cart_changes['current_cart_rule']).mean()*100:.1f}%")

# Display sample
print(f"\n{'='*60}")
print("SAMPLE OUTPUT")
print(f"{'='*60}")
output_cols = ['sku', 'current_price', 'new_price', 'current_cart_rule', 'new_cart_rule', 
               'price_action', 'cart_rule_action', 'action_reason']
output_cols = [c for c in output_cols if c in df_action.columns]
display(df_action[df_action['new_price'].notna() | df_action['new_cart_rule'].notna()][output_cols].head(15))


In [ ]:
# =============================================================================
# FIXED PRICE OVERRIDE (from Google Sheet) - skip WAC increase actions
# =============================================================================
df_fixed = get_fixed_prices()
df_action = df_action.merge(df_fixed, on='product_id', how='left')
has_fixed = df_action['fixed_price'].notna()
not_wac = df_action['price_action'] != 'wac_increase'
apply_fixed = has_fixed & not_wac
df_action.loc[apply_fixed, 'new_price'] = df_action.loc[apply_fixed, 'fixed_price']
df_action.loc[apply_fixed, 'price_action'] = 'fixed_price'
df_action = df_action.drop(columns=['fixed_price'])
print(f"Fixed price override: {apply_fixed.sum()} SKUs set to fixed price from Google Sheet")
if (has_fixed & ~not_wac).sum() > 0:
    print(f"  Skipped {(has_fixed & ~not_wac).sum()} fixed-price SKUs due to WAC increase")

# =============================================================================
# FIXED CART RULE OVERRIDE (from Google Sheet - Sheet2)
# =============================================================================
df_fixed_cart = get_fixed_cart_rules()
df_action = df_action.merge(df_fixed_cart, on='product_id', how='left')
has_fixed_cart = df_action['fixed_cart_rule'].notna()
df_action.loc[has_fixed_cart, 'new_cart_rule'] = df_action.loc[has_fixed_cart, 'fixed_cart_rule'].astype(int)
df_action = df_action.drop(columns=['fixed_cart_rule'])
print(f"Fixed cart rule override: {has_fixed_cart.sum()} SKUs set to fixed cart rule from Google Sheet")

In [ ]:
# =============================================================================
# PUSH CONFIGURATION
# =============================================================================
# Mode: 'testing' = prepare files only, 'live' = actually upload to API
PUSH_MODE = 'live'  # Change to 'live' when ready to push

print(f"\n{'='*60}")
print(f"PUSH MODE: {PUSH_MODE.upper()}")
print(f"{'='*60}")
if PUSH_MODE == 'testing':
    print("⚠️ Testing mode - files will be prepared but NOT uploaded")
else:
    print("🚀 Live mode - files will be uploaded to MaxAB API")


In [ ]:
# Save df_action state before any manipulation for Slack upload later
temp_df_for_slack = df_action.copy()
print(f"✅ Saved {len(temp_df_for_slack)} rows for Slack upload")

In [ ]:
# =============================================================================
# IMPORT PUSH HANDLERS
# =============================================================================
%run push_cart_rules_handler.ipynb
%run push_prices_handler.ipynb

print("✅ Push handlers loaded")


In [ ]:
# =============================================================================
# PREPARE DATA FOR PUSH
# =============================================================================
# Get packing units for push handlers
pus = get_packing_units()
print(f"✅ Loaded {len(pus)} packing units")

# Prepare df_action with required columns for push functions
# Prices need: product_id, sku, new_price, warehouse_id, cohort_id, stocks, current_price
# Cart rules need: product_id, sku, new_cart_rule, warehouse_id, cohort_id, stocks, current_cart_rule

# Rename stocks column if needed
if 'current_stocks' in df_action.columns and 'stocks' not in df_action.columns:
    df_action['stocks'] = df_action['current_stocks']

# Drop duplicates to prevent double-pushes from merge row multiplication
before_dedup = len(df_action)
df_action = df_action.drop_duplicates(subset=['warehouse_id', 'product_id',], keep='first')
print(f"After dedup: {len(df_action)} unique SKU actions (removed {before_dedup - len(df_action)} duplicates)")

# Summary of what will be pushed
price_changes = df_action[df_action['new_price'].notna() & (df_action['new_price'] != df_action['current_price'])]
cart_changes = df_action[df_action['new_cart_rule'].notna() & (df_action['new_cart_rule'] != df_action['current_cart_rule'])]

print(f"\n📊 Push Summary:")
print(f"  Price changes: {len(price_changes)} SKUs")
print(f"  Cart rule changes: {len(cart_changes)} SKUs")
print(f"  Mode: {PUSH_MODE}")


In [ ]:
# =============================================================================
# STEP 1: PUSH CART RULES
# =============================================================================
# Push cart rules first - if any cohorts fail, we'll skip them for prices too
print(f"\n{'='*60}")
print("STEP 1: PUSH CART RULES")
print(f"{'='*60}")

cart_result = push_cart_rules(df_action, pus, source_module='module_4', mode=PUSH_MODE)

# Track failed cohorts to skip in price push
failed_cohorts = cart_result.get('failed_cohorts', [])

print(f"\n📊 Cart Rules Push Result:")
print(f"  Total received: {cart_result['total_received']}")
print(f"  Cart rule changes: {cart_result['cart_rule_changes']}")
print(f"  Pushed: {cart_result['pushed']}")
print(f"  Skipped: {cart_result['skipped']}")
print(f"  Failed: {cart_result['failed']}")
print(f"  Mode: {cart_result['mode']}")

if failed_cohorts:
    print(f"  ⚠️ Failed cohorts: {failed_cohorts}")


In [ ]:
# =============================================================================
# STEP 2: PUSH PRICES (skip failed cohorts from cart rules)
# =============================================================================
print(f"\n{'='*60}")
print("STEP 2: PUSH PRICES")
print(f"{'='*60}")

# Push prices, skipping any cohorts that failed during cart rules push
push_result = push_prices(df_action, pus, source_module='module_4', mode=PUSH_MODE, skip_cohorts=failed_cohorts)

print(f"\n📊 Prices Push Result:")
print(f"  Total received: {push_result['total_received']}")
print(f"  Price changes: {push_result['price_changes']}")
print(f"  Pushed: {push_result['pushed']}")
print(f"  Skipped: {push_result['skipped']}")
print(f"  Failed: {push_result['failed']}")
print(f"  Mode: {push_result['mode']}")

if push_result.get('skipped_cohorts'):
    print(f"  ⚠️ Skipped cohorts: {push_result['skipped_cohorts']}")


In [ ]:
# =============================================================================
# FINAL SUMMARY
# =============================================================================
print(f"\n{'='*60}")
print("MODULE 4 - HOURLY UPDATES COMPLETE")
print(f"{'='*60}")
print(f"\n📅 Timestamp: {datetime.now(CAIRO_TZ).strftime('%Y-%m-%d %H:%M:%S')}")
print(f"🔧 Mode: {PUSH_MODE.upper()}")

print(f"\n📊 ACTIONS TAKEN:")
print(f"  Total SKUs analyzed: {len(df)}")
print(f"  SKUs requiring action: {len(df_action)}")

# Price actions breakdown
print(f"\n💰 PRICE ACTIONS:")
if 'price_action' in df_action.columns:
    price_actions = df_action[df_action['price_action'] != 'none']
    print(f"  Total price changes: {len(price_actions)}")
    for action_type in df_action['price_action'].unique():
        if action_type != 'none':
            count = len(df_action[df_action['price_action'] == action_type])
            print(f"    - {action_type}: {count}")
else:
    print(f"  Total price changes: 0")

# Cart rule actions breakdown  
print(f"\n🛒 CART RULE ACTIONS:")
if 'cart_rule_action' in df_action.columns:
    cart_actions = df_action[df_action['cart_rule_action'] != 'none']
    print(f"  Total cart rule changes: {len(cart_actions)}")
    for action_type in df_action['cart_rule_action'].unique():
        if action_type != 'none':
            count = len(df_action[df_action['cart_rule_action'] == action_type])
            print(f"    - {action_type}: {count}")
else:
    print(f"  Total cart rule changes: 0")

# Push results
print(f"\n📤 PUSH RESULTS:")
print(f"  Cart Rules - Pushed: {cart_result.get('pushed', 0)}, Failed: {cart_result.get('failed', 0)}")
print(f"  Prices - Pushed: {push_result.get('pushed', 0)}, Failed: {push_result.get('failed', 0)}")

if PUSH_MODE == 'testing':
    print(f"\n⚠️ TESTING MODE - No changes were actually pushed to production!")
    print(f"   Change PUSH_MODE to 'live' to execute actual pushes.")
else:
    print(f"\n✅ LIVE MODE - Changes have been pushed to production!")


In [ ]:
# =============================================================================
# SAMPLE OUTPUT - Current Status
# =============================================================================
# Show sample of data with all calculated fields

sample_cols = [
    'warehouse_id', 'product_id', 'sku','wac1','wac_p','new_wac',
    # P80/P70 benchmarks and std
    'p80_daily_240d', 'std_daily_240d', 'p70_daily_retailers_240d', 'std_daily_retailers_240d',
    # Current cart rule
    'current_cart_rule',
    # UTH performance (with std thresholds)
    'uth_qty', 'uth_qty_target', 'uth_qty_std', 'uth_qty_status',
    'uth_retailers', 'uth_rets_target', 'uth_rets_std', 'uth_rets_status',
    # Last hour performance (with std thresholds)
    'last_hour_qty', 'last_hour_qty_target', 'last_hour_qty_std', 'last_hour_qty_status',
    'last_hour_retailers', 'last_hour_rets_target', 'last_hour_rets_std', 'last_hour_rets_status'
]

# Filter to columns that exist
sample_cols = [c for c in sample_cols if c in df.columns]

print(f"\n{'='*60}")
print("SAMPLE DATA (First 10 rows with UTH > 0)")
print(f"{'='*60}")
sample = df[df['uth_qty'] > 0][sample_cols].head(10)
display(sample)


In [ ]:
push_columns = ['warehouse_id','product_id','cohort_id','sku','region','brand','cat','current_price','wac_p','new_wac','current_margin','stocks','current_cart_rule','p80_daily_240d','p70_daily_retailers_240d','std_daily_240d','std_daily_retailers_240d','normal_refill','refill_stddev','target_margin','uth_qty','uth_qty_status','uth_retailers','uth_rets_status','last_hour_qty','last_hour_qty_status','last_hour_retailers','last_hour_rets_status','below_market','market_min','market_25','market_50','market_75','market_max','above_market','margin_tier_1','margin_tier_2','margin_tier_3','margin_tier_4','margin_tier_5','margin_tier_above_1','margin_tier_above_2','needs_action','action_reason','normal_refill_3std','new_price','new_cart_rule','price_action','cart_rule_action','market_min_price','market_25_price','market_50_price','market_75_price','market_max_price','margin_tier_1_price','margin_tier_2_price','margin_tier_3_price','margin_tier_4_price','margin_tier_5_price','margin_tier_above_1_price','margin_tier_above_2_price']
df_action = df_action[push_columns]

In [ ]:
# =============================================================================
# UPLOAD RESULTS TO SNOWFLAKE AND SEND SLACK NOTIFICATION
# =============================================================================

from common_functions import upload_dataframe_to_snowflake, send_text_slack, send_file_slack

SLACK_CHANNEL_ID = 'C0AAWK97Z3Q'

# Add created_at as TIMESTAMP (module runs hourly)
df_action['created_at'] = datetime.now(CAIRO_TZ).replace(second=0, microsecond=0)
# Drop columns not in database schema (internal columns used for logic only)
df_action = df_action.drop(columns=['commercial_min_price', 'doh', 'qty_price_step_count','qtr_cntrb','uth_qty_target'], errors='ignore')
# Upload to Snowflake
print(f"\n{'='*60}")
print("UPLOADING RESULTS TO SNOWFLAKE")
print(f"{'='*60}")

upload_status = upload_dataframe_to_snowflake(
    "Egypt", 
    df_action, 
    "MATERIALIZED_VIEWS", 
    "pricing_hourly_push", 
    "append", 
    auto_create_table=True, 
    conn=None
)

# Prepare status variables
prices_pushed = push_result.get('pushed', 0) if 'push_result' in dir() else 0
prices_failed = push_result.get('failed', 0) if 'push_result' in dir() else 0
cart_rules_pushed = cart_result.get('pushed', 0) if 'cart_result' in dir() else 0
cart_rules_failed = cart_result.get('failed', 0) if 'cart_result' in dir() else 0

# Count price and cart rule actions
price_changes = len(df_action[df_action['price_action'] != 'none']) if 'price_action' in df_action.columns else 0
cart_changes = len(df_action[df_action['cart_rule_action'] != 'none']) if 'cart_rule_action' in df_action.columns else 0

if upload_status:
    slack_message = f"""✅ *Module 4 - Hourly Updates Completed*

📅 Date: {datetime.now(CAIRO_TZ).strftime('%Y-%m-%d')}
⏰ Hour: {datetime.now(CAIRO_TZ).strftime('%H:%M:%S')} Cairo time
🔧 Mode: {PUSH_MODE.upper()}

📊 *Results:*
• Total SKUs analyzed: {len(df):,}
• SKUs requiring action: {len(df_action):,}
• Price changes: {price_changes:,}
• Cart rule changes: {cart_changes:,}

📤 *Push Status:*
• 💰 Prices: ✅ {prices_pushed} pushed | ❌ {prices_failed} failed
• 🛒 Cart Rules: ✅ {cart_rules_pushed} pushed | ❌ {cart_rules_failed} failed

🗄️ Results uploaded to: MATERIALIZED_VIEWS.pricing_hourly_push"""
    
    send_text_slack('new-pricing-logic', slack_message)
    print("✅ Slack notification sent!")
    
    # Send review file to Slack after the text message (using saved copy before manipulation)
    send_file_slack(
        temp_df_for_slack, 
        f'📎 Module 4 Review: {len(temp_df_for_slack)} SKUs processed', 
        SLACK_CHANNEL_ID,
        filename=f'module4_hourly_{datetime.now(CAIRO_TZ).strftime("%Y%m%d_%H%M")}.xlsx'
    )
    print("✅ Review file sent to Slack")
    
    print(f"✅ {len(df_action)} records uploaded to Snowflake")
else:
    error_message = f"""❌ *Module 4 - Hourly Updates Failed*

📅 Date: {datetime.now(CAIRO_TZ).strftime('%Y-%m-%d')}
⏰ Hour: {datetime.now(CAIRO_TZ).strftime('%H:%M:%S')} Cairo time
⚠️ Upload to Snowflake failed - please check logs

📤 *Push Status (before upload failure):*
• 💰 Prices: ✅ {prices_pushed} pushed | ❌ {prices_failed} failed
• 🛒 Cart Rules: ✅ {cart_rules_pushed} pushed | ❌ {cart_rules_failed} failed"""
    
    send_text_slack('new-pricing-logic', error_message)
    print("❌ Error notification sent to Slack!")
    
    # Still send review file even on error for debugging (using saved copy before manipulation)
    send_file_slack(
        temp_df_for_slack, 
        f'⚠️ Module 4 ERROR Review: {len(temp_df_for_slack)} SKUs', 
        SLACK_CHANNEL_ID,
        filename=f'module4_hourly_ERROR_{datetime.now(CAIRO_TZ).strftime("%Y%m%d_%H%M")}.xlsx'
    )
    print("✅ Error review file sent to Slack")
